In [1]:
import sys
import os
import io
import base64
import datetime
from PIL import Image
from ollama import chat

sys.path.append(os.path.abspath(os.path.join('..')))

from src.video.extractor import get_video_metadata, chunk_video, extract_frames
from src.config import settings


In [2]:
VIDEO_PATH = "../data/input/videoplayback.mp4"
fps, _, _, frame_count = get_video_metadata(VIDEO_PATH)
all_idxs = chunk_video(
        settings.chunk_t, settings.chunk_frames, settings.overlap, fps, frame_count
    )
stream = extract_frames(VIDEO_PATH, all_idxs)

print(all_idxs)
first_chunk = next(stream)
first_frame = first_chunk[0]
print(first_frame)

[[0, 99, 198, 298], [240, 339, 438, 538], [480, 579, 678, 778], [720, 819, 918, 1018], [960, 1059, 1158, 1258], [1200, 1275, 1350, 1426]]
[[[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 ...

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]

 [[0 0 0]
  [0 0 0]
  [0 0 0]
  ...
  [0 0 0]
  [0 0 0]
  [0 0 0]]]


In [ ]:
img = Image.fromarray(first_frame)
buffered = io.BytesIO()
img.save(buffered, format="JPEG")
img_bytes = buffered.getvalue()
temp_frame = base64.b64encode(img_bytes).decode('utf-8')

In [ ]:
#client = Client()

messages = [
    {
        'role': 'user', 
        'content': 'What is in this image', 
        'images': [temp_frame]
    }
]

print("User: What is in this image")
print("Assistant:")

try:
    response = chat(model='gemma4:e2b', messages=messages, think = 'low')
    print(response.message.content)
except Exception as e:
    print(f"An error occurred: {e}")

### Full chunk handling

In [ ]:
for chunk_idx, chunk in enumerate(stream):
    chunks_frames = []
    frame_indices = []
    for frame_idx, frame in enumerate(chunk):
        img = Image.fromarray(frame)
        buffered = io.BytesIO()
        img.save(buffered, format="JPEG")
        img_bytes = buffered.getvalue()
        chunks_frames.append(base64.b64encode(img_bytes).decode('utf-8'))
        frame_indices.append(frame_idx)
    print(chunks_frames)
    print(frame_indices)

['...']
[0, 1, 2, 3]
['...']
[0, 1, 2, 3]
['...']
[0, 1, 2, 3]
['...']
[0, 1, 2, 3]
['...']
[0, 1, 2, 3]


In [150]:
def build_prompt(all_idxs):
    lines = [
        "You are given a sequence of images.",
        "The images are provided in the exact order listed below.",
        "Image N corresponds to the N-th image in the input.",
        "",
    ]

    for i, j in enumerate(all_idxs, 1):
        raw_timestamp = j / fps
        timestamp = str(datetime.timedelta(seconds=raw_timestamp))
        lines.append(f"Image {i}: timestamp {timestamp}")

    lines.extend([
        "",
        "Analyze the sequence in chronological order.",
        "For each image:",
        "- describe the scene,",
        "- identify changes from the previous image,",
        "- infer the ongoing activity.",
    ])

    return "\n".join(lines)

In [151]:
def convert_tobase(chunk):
    chunks_frames = []
    for _, frame in enumerate(chunk):
        img = Image.fromarray(frame)
        buffered = io.BytesIO()
        img.save(buffered, format="JPEG")
        img_bytes = buffered.getvalue()
        chunks_frames.append(base64.b64encode(img_bytes).decode('utf-8'))
        yield chunks_frames

In [152]:
for chunk_idx, chunk in enumerate(stream):

    chunks_frames = convert_tobase(chunk)

    messages = [{
        "role": "user",
        "content": build_prompt(all_idxs[chunk_idx]),
        "images": [img for img in chunks_frames]
    }]

    print(messages)

    #try:
    #    response = chat(model='gemma4:e2b', messages=messages, think = 'low')
    #    print(response.message.content)
    #except Exception as e:
    #    print(f"An error occurred: {e}")

[{'role': 'user', 'content': 'You are given a sequence of images.\nThe images are provided in the exact order listed below.\nImage N corresponds to the N-th image in the input.\n\nImage 1: timestamp 0:00:00\nImage 2: timestamp 0:00:03.303300\nImage 3: timestamp 0:00:06.606600\nImage 4: timestamp 0:00:09.943267\n\nAnalyze the sequence in chronological order.\nFor each image:\n- describe the scene,\n- identify changes from the previous image,\n- infer the ongoing activity.', 'images': [['...']]}]
[{'role': 'user', 'content': 'You are given a sequence of images.\nThe images are provided in the exact order listed below.\nImage N corresponds to the N-th image in the input.\n\nImage 1: timestamp 0:00:08.008000\nImage 2: timestamp 0:00:11.311300\nImage 3: timestamp 0:00:14.614600\nImage 4: timestamp 0:00:17.951267\n\nAnalyze the sequence in chronological order.\nFor each image:\n- describe the scene,\n- identify changes from the previous image,\n- infer the ongoing activity.', 'images': [['..